# SecureOps Assistant — End-to-End Pipeline

**Vector Cartel** · AAI Tech Talks Hackathon 2026 · WMG, University of Warwick

A RAG-based ICS cybersecurity assistant. This notebook runs the full pipeline end to end:

```
user query
   -> classify_query        (llm-and-agentic)
   -> retrieve               (rag-layer: ChromaDB + BM25 + cross-encoder rerank
                              over NIST SP 800-82, CISA advisories, MITRE ATT&CK for ICS)
   -> synthesize_answer      (llm-and-agentic: Gemini, cited inline)
   -> validate_citations     (llm-and-agentic: token-overlap groundedness check)
   -> SecureOpsAnswer        (formatted with numbered sources)
```

Run the cells top to bottom. You'll need a **Gemini API key** (free tier) and,
for the fallback provider, a **HuggingFace API key** — see Setup below.

## 1. Setup

Install dependencies for both layers (LLM/agent + RAG retrieval). Skip this cell
if you're running locally and already have these installed.

In [ ]:
%pip install -q -r ../requirements-llm.txt -r ../requirements.txt

### API keys

Set `GEMINI_API_KEY` (required) and `HF_API_KEY` (required, used as fallback
provider if Gemini's free-tier rate limit is hit). In Colab, use **Secrets**
(the key icon in the left sidebar) and uncomment the `userdata` block below.
Locally, set them as environment variables before launching Jupyter, or enter
them via the prompts below.

In [ ]:
import os
import getpass

# --- Colab Secrets (uncomment if running in Colab) ---
# from google.colab import userdata
# os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
# os.environ["HF_API_KEY"] = userdata.get("HF_API_KEY")

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
if not os.environ.get("HF_API_KEY"):
    os.environ["HF_API_KEY"] = getpass.getpass("HF_API_KEY: ")

print("API keys set.")

### Make `src/` importable

If running from inside `notebooks/`, add the repo root to `sys.path` so
`from src.x import y` imports resolve the same way they do for `pytest`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root on sys.path: {REPO_ROOT}")

## 2. Initialise the pipeline

`setup()` (src/gradio_demo.py) wires everything together:
- `LLMRouter` — Gemini 1.5 Flash primary, HuggingFace Mistral-7B fallback
- `retrieval_fn` — rag-layer's real `build_retrieval_fn()`: loads the
  pre-built ChromaDB + BM25 indexes over the ICS corpus (NIST SP 800-82,
  CISA advisories, MITRE ATT&CK for ICS) and a cross-encoder reranker

This takes ~10-30s the first time (downloading/loading the embedding and
reranker models).

In [ ]:
from src.gradio_demo import setup, chat_fn, format_response
from src.agent import run_agent

llm_router, retrieval_fn = setup()
print("Pipeline ready: llm_router + retrieval_fn initialised.")

## 3. End-to-end demo queries

Each call below runs the **full graph**: classify -> (decompose if multi-hop)
-> retrieve -> verify -> synthesize -> validate -> `SecureOpsAnswer`.

### 3a. Simple factual query

In [ ]:
result = run_agent(
    "How should firewalls segment OT networks from IT networks?",
    llm_router,
    retrieval_fn,
)

print(format_response(
    answer=result.answer,
    citations=result.citations,
    refusal=result.refusal,
))
print(f"\n[confidence: {result.confidence:.2f} | query_type: {result.query_type}]")

### 3b. Multi-hop query (decomposed across knowledge sources)

In [ ]:
result = run_agent(
    "What NIST controls defend against the techniques used in recent "
    "Siemens advisories?",
    llm_router,
    retrieval_fn,
)

print(format_response(
    answer=result.answer,
    citations=result.citations,
    refusal=result.refusal,
))
print(f"\n[confidence: {result.confidence:.2f} | query_type: {result.query_type}]")

### 3c. Out-of-scope query — the honesty test

SecureOps Assistant only answers from its corpus. A question about *your*
company's internal configuration is out of scope and must be refused, not
hallucinated.

In [ ]:
result = run_agent(
    "What is our company's firewall configuration?",
    llm_router,
    retrieval_fn,
)

print(format_response(
    answer=result.answer,
    citations=result.citations,
    refusal=result.refusal,
))
print(f"\n[refusal: {result.refusal} | confidence: {result.confidence:.2f}]")

## 4. Interactive demo (optional)

Launches a Gradio chat interface so judges can type their own queries live.

In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=lambda message, history: chat_fn(message, history, llm_router, retrieval_fn),
    title="SecureOps Assistant",
    description=(
        "ICS cybersecurity Q&A grounded in NIST SP 800-82, CISA advisories, "
        "and MITRE ATT&CK for ICS. Vector Cartel · AAI Tech Talks Hackathon 2026."
    ),
)
demo.launch(share=False)